# Lecture 10 — `std::optional`

Today’s goal:

> Represent “maybe there is a value, maybe not” safely.

This is useful for metric calculations that can fail.

## 1. What is `std::optional`?

```cpp
std::optional<double> maybe_score;
```

Meaning:

```text
Maybe it contains a double.
Maybe it contains nothing.
```

To use it:

```cpp
#include <optional>
```

In [1]:
#include <iostream>
#include <optional>

{
  std::optional<double> maybe_score;

  std::cout << std::boolalpha;
  std::cout << "has value = " << maybe_score.has_value() << std::endl;
}

input_line_8:3:8: error: no member named 'optional' in namespace 'std'
  std::optional<double> maybe_score;
  ~~~~~^
input_line_8:3:23: error: expected '(' for function-style cast or type construction
  std::optional<double> maybe_score;
                ~~~~~~^
input_line_8:3:25: error: use of undeclared identifier 'maybe_score'
  std::optional<double> maybe_score;
                        ^
input_line_8:5:34: error: use of undeclared identifier 'maybe_score'
  std::cout << "has value = " << maybe_score.has_value() << std::endl;
                                 ^


Interpreter Error: 

## 2. Returning `std::nullopt`

`std::nullopt` means no value.

In [2]:
#include <iostream>
#include <optional>

std::optional<double> divide_10(double a, double b)
{
  if (b == 0.0) {
    return std::nullopt;
  }

  return a / b;
}

{
  auto result = divide_10(10.0, 0.0);

  if (result.has_value()) {
    std::cout << "value = " << result.value() << std::endl;
  } else {
    std::cout << "division failed" << std::endl;
  }
}

input_line_10:1:6: error: no template named 'optional' in namespace 'std'
std::optional<double> divide_10(double a, double b)
~~~~~^
input_line_10:4:17: error: no member named 'nullopt' in namespace 'std'
    return std::nullopt;
           ~~~~~^
In file included from input_line_5:1:
In file included from /home/user/miniconda3/envs/cling/include/xeus/xinterpreter.hpp:17:
In file included from /home/user/miniconda3/envs/cling/include/xeus/xcomm.hpp:19:
In file included from /home/user/miniconda3/envs/cling/include/nlohmann/json.hpp:40:
In file included from /home/user/miniconda3/envs/cling/include/nlohmann/detail/input/binary_reader.hpp:25:
In file included from /home/user/miniconda3/envs/cling/include/nlohmann/detail/input/input_adapters.hpp:23:
In file included from /home/user/miniconda3/envs/cling/bin/../lib/gcc/../../x86_64-conda-linux-gnu/include/c++/10.4.0/istream:39:
/home/user/miniconda3/envs/cling/bin/../lib/gcc/../../x86_64-conda-linux-gnu/include/c++/10.4.0/ostream:609:8: er

Interpreter Error: 

## 3. `.has_value()` and `.value()`

```cpp
if (result.has_value()) {
  double x = result.value();
}
```

Never call `.value()` before checking if value exists.

In [3]:
#include <iostream>
#include <optional>

{
  std::optional<double> maybe_distance_m = 12.5;

  if (maybe_distance_m.has_value()) {
    std::cout << "distance = " << maybe_distance_m.value() << std::endl;
  }
}

input_line_12:3:8: error: no member named 'optional' in namespace 'std'
  std::optional<double> maybe_distance_m = 12.5;
  ~~~~~^
input_line_12:3:23: error: expected '(' for function-style cast or type construction
  std::optional<double> maybe_distance_m = 12.5;
                ~~~~~~^
input_line_12:3:25: error: use of undeclared identifier 'maybe_distance_m'
  std::optional<double> maybe_distance_m = 12.5;
                        ^
input_line_12:4:7: error: use of undeclared identifier 'maybe_distance_m'
  if (maybe_distance_m.has_value()) {
      ^
input_line_12:5:35: error: use of undeclared identifier 'maybe_distance_m'
    std::cout << "distance = " << maybe_distance_m.value() << std::endl;
                                  ^
In file included from input_line_5:1:
In file included from /home/user/miniconda3/envs/cling/include/xeus/xinterpreter.hpp:17:
In file included from /home/user/miniconda3/envs/cling/include/xeus/xcomm.hpp:19:
In file included from /home/user/miniconda3/envs/

Interpreter Error: 

## 4. Metric-style use case

Suppose progress calculation requires at least two points. If not, return no value.

In [ ]:
#include <iostream>
#include <optional>
#include <vector>

std::optional<double> calculate_raw_progress_10(const std::vector<double> & x_positions)
{
  if (x_positions.size() < 2) {
    return std::nullopt;
  }

  return x_positions.back() - x_positions.front();
}

{
  std::vector<double> x_positions{0.0};
  auto progress = calculate_raw_progress_10(x_positions);

  if (!progress.has_value()) {
    std::cout << "unavailable_short_trajectory" << std::endl;
  } else {
    std::cout << "progress = " << progress.value() << std::endl;
  }
}

## 5. Optional + MetricResult

This pattern is common:

```cpp
const auto raw = calculate_something(...);
if (!raw.has_value()) {
  result.reason = "unavailable_xxx";
  return result;
}
result.raw_value = raw.value();
```

In [ ]:
#include <iostream>
#include <optional>
#include <string>
#include <vector>

struct ProgressResult10
{
  double score{0.0};
  bool available{false};
  std::string reason{"unavailable"};
  double raw_progress_m{0.0};
};

std::optional<double> calculate_raw_progress_helper_10(const std::vector<double> & xs)
{
  if (xs.size() < 2) {
    return std::nullopt;
  }
  return xs.back() - xs.front();
}

ProgressResult10 calculate_progress_metric_10(const std::vector<double> & xs)
{
  ProgressResult10 result;

  const auto raw_progress = calculate_raw_progress_helper_10(xs);
  if (!raw_progress.has_value()) {
    result.reason = "unavailable_short_trajectory";
    return result;
  }

  result.available = true;
  result.score = 1.0;
  result.reason = "available";
  result.raw_progress_m = raw_progress.value();
  return result;
}

{
  const auto result = calculate_progress_metric_10({0.0, 3.0});

  std::cout << std::boolalpha;
  std::cout << "available = " << result.available << std::endl;
  std::cout << "score = " << result.score << std::endl;
  std::cout << "reason = " << result.reason << std::endl;
  std::cout << "raw_progress_m = " << result.raw_progress_m << std::endl;
}

## 6. Homework

Write a helper:

```cpp
std::optional<double> get_first_speed(const std::vector<double> & speeds)
```

Return `std::nullopt` if empty, otherwise return the first speed.

In [ ]:
#include <iostream>
#include <optional>
#include <vector>

std::optional<double> get_first_speed_10(const std::vector<double> & speeds)
{
  // TODO: implement.
  return std::nullopt;
}

{
  std::vector<double> speeds{3.0, 5.0};
  auto first = get_first_speed_10(speeds);

  if (first.has_value()) {
    std::cout << "first speed = " << first.value() << std::endl;
  } else {
    std::cout << "no speed" << std::endl;
  }
}

Next lecture: **`.hpp` and `.cpp`** — how C++ code is organized across files.